In [1]:
clean_up = True # if True, remove all gams related files from working folder before starting
%run stdPackages.ipynb

The file _gams_py_gdb0.gdx is still active and was not deleted.
The file _gams_py_gdb1.gdx is still active and was not deleted.
The file _gams_py_gdb10.gdx is still active and was not deleted.
The file _gams_py_gdb11.gdx is still active and was not deleted.
The file _gams_py_gdb12.gdx is still active and was not deleted.
The file _gams_py_gdb13.gdx is still active and was not deleted.
The file _gams_py_gdb14.gdx is still active and was not deleted.
The file _gams_py_gdb15.gdx is still active and was not deleted.
The file _gams_py_gdb16.gdx is still active and was not deleted.
The file _gams_py_gdb17.gdx is still active and was not deleted.
The file _gams_py_gdb18.gdx is still active and was not deleted.
The file _gams_py_gdb19.gdx is still active and was not deleted.
The file _gams_py_gdb2.gdx is still active and was not deleted.
The file _gams_py_gdb20.gdx is still active and was not deleted.
The file _gams_py_gdb21.gdx is still active and was not deleted.
The file _gams_py_gdb22.gdx 

*Load specific modules used here:*

In [2]:
os.chdir(d['py'])
import mCGE 
os.chdir(os.path.join(d['curr'], 'py'))
import report

# Shocks

Load base model and data:

In [3]:
t0 = 2019
name = f'vGRSIntRC{t0}CGE'
M = mCGE.WasteManagementCGE.load(os.path.join(d['data'], name)) # load model
ws = M.ws 
db0 = M.db.copy() # store initial solution

Initialize reporting class:

In [4]:
Rep = report.Standard(db0)

Add baseline report:

In [5]:
d0 = Rep(db0, {})
d0['db'] = db0

Empty dictionary to store solutions in:

In [6]:
shocks = {'Baseline': d0} # Dictionary to store shocks in

## 1. Increase in internal recycling rate

### A. Shock to metal:

Define shock:

In [7]:
def exp_IRRmetal(Mi, solDict):
    ξnew = adj.rc_pd(Mi.db('intRcEff'), pd.Index(['Metal'],name='m'))+0.1
    μWP0 = (Mi.db('uWS_int') * (Mi.db('qWS')/(1-Mi.db('uWS_int') * Mi.db('intRcEff')))).dropna() # μint * W^P in baseline
    solDict['shock'] = ξnew
    solDict['Δexp'] = -((ξnew-Mi.db('intRcEff')) * μWP0).dropna().groupby(['t','m']).sum() # expected decrease in use of virgin materials
    return solDict

Add solution dictionary:

In [8]:
name = 'IRRmetal'
shocks[name] = exp_IRRmetal(M, {})
di = shocks[name]

Solve with shock:

In [9]:
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state='B')

Report solution:

In [10]:
di = Rep(di['db'], di, d0 = d0)

Revert model data again:

In [11]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']), priority = 'second')

### B. Shock to plastic:

In [12]:
def exp_IRRplastic(Mi, solDict):
    ξnew = adj.rc_pd(Mi.db('intRcEff'), pd.Index(['Plastic'],name='m'))+0.1
    μWP0 = (Mi.db('uWS_int') * (Mi.db('qWS')/(1-Mi.db('uWS_int') * Mi.db('intRcEff')))).dropna() # μint * W^P in baseline
    solDict['shock'] = ξnew
    solDict['Δexp'] = -((ξnew-Mi.db('intRcEff')) * μWP0).dropna().groupby(['t','m']).sum() # expected decrease in use of virgin materials
    return solDict

Add solution dictionary:

In [13]:
name = 'IRRplastic'
shocks[name] = exp_IRRplastic(M, {})
di = shocks[name]

Solve with shock:

In [14]:
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state='B')

Report solution:

In [15]:
di = Rep(di['db'], di, d0 = d0)

Revert model data again:

In [16]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']), priority = 'second')

### C. Shock to all

In [17]:
def exp_IRRall(Mi, solDict):
    ξnew = Mi.db('intRcEff')+0.1
    μWP0 = (Mi.db('uWS_int') * (Mi.db('qWS')/(1-Mi.db('uWS_int') * Mi.db('intRcEff')))).dropna() # μint * W^P in baseline
    solDict['shock'] = ξnew
    solDict['Δexp'] = -((ξnew-Mi.db('intRcEff')) * μWP0).dropna().groupby(['t','m']).sum() # expected decrease in use of virgin materials
    return solDict

Add solution dictionary:

In [18]:
name = 'IRRall'
shocks[name] = exp_IRRall(M, {})
di = shocks[name]

Solve with shock:

In [19]:
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state='B')

Report solution:

In [20]:
di = Rep(di['db'], di, d0 = d0)

Revert model data again:

In [21]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']), priority = 'second')

## 2. Changes in recycling technology in waste management

In [22]:
def f_intApprox(x, l = 0, u = 1, eps = 0.0001):
    return (l+np.sqrt(np.square(x-l)+eps)+u-np.sqrt(np.square(x-u)+eps))/2
def f_techGInv(y, l = 0, u = 1, b = 2):
    return ((u-y)/(u-l)).pow(1/b)
def f_techF(r, l = 0, u = 1, b = 2):
    return l+(u-l)*(1-r.pow(b)/(1+b))

### A. Define shock and compute mechanical effect

In [23]:
name = 'RCEff' # Recycling efficiency 
shocks[name] = {}
di = shocks[name] 

Increase $\underline{\alpha}$ in recycling technology for metal and compute mechanical effect of shock:

In [24]:
new_αl = M.db('WTD_alphal')+0.1
# new_αl = adj.rc_pd(M.db('WTD_alphal'), pd.Index(['Metal'], name = 'm'))+0.1
di['shock']= new_αl
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()

Given prices and scale (amount of waste to be treated), how much extra recycling do we get out of this shock?

In [25]:
l, u, b, eps = M.db('WTD_alphal'), M.db('WTD_alphau'), M.db('WTD_beta'), M.db('WTD_smooth')
y = f_intApprox(M.db('WTD_zetaE'), l = l, u = u, eps = eps) 
r = f_techGInv(y, l= l, u = u, b = b) # R/R̄
a = f_techF(r, l = l, u = u, b = b) # A/R
R = M.db('WTD_W') * r # R
A = (a * R).dropna() # A

What is the reduction in virgin materials compared to baseline, if they reduce this 1-to-1:

In [26]:
qSR0 = adjMultiIndex.applyMult(d0['db']('qS').xs('Waste',level='s'), db0('nr2m_D')).droplevel('n').dropna() # supply of recycled materials in baseline, mapped to 'm' index
di['Δexp'] = -(A-qSR0).dropna()
# di['Δexp'] = -adj.rc_pd((A-qSR0).dropna(), pd.Index(['Metal'], name = 'm'))

Solve:

In [27]:
di['db'] = M.solve(state = 'B')

Report solution:

In [28]:
di = Rep(di['db'], di, d0 = d0)

Revert model data:

In [29]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

## 3. Tax on virgin materials

In [30]:
name = 'taxVirgin' # Recycling efficiency 
shocks[name] = {}
di = shocks[name] 

Shock:

In [31]:
di['shock'] = adj.rc_pd(M.db('tauD'), Rep.vm)+0.1
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state = 'B')

Report solution:

In [32]:
di = Rep(di['db'], di, d0 = d0)

Revert model data:

In [33]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

## 4. Subsidy to recycling/mandate to increase recycling

In [34]:
name = 'RCmandate' # Recycling efficiency 
shocks[name] = {}
di = shocks[name] 

Assume that recycling of textile increases from 27 to 50%. The expected effect of this is given by:

In [35]:
targetRate = 0.5
mi = pd.Index(['Textile'], name = 'm')
sIdx = pd.Index(['Waste'],name='s') # relevant industry
ni = adjMultiIndex.applyMult(mi, M.db('nr2m_D')) 
R̄ = adj.rc_pd((M.db('WTD_W')*(1-M.db('WTD_dmin'))).dropna(), mi)
Rtarget = R̄ * targetRate
di['Δexp'] = (R̄ * adj.rc_pd(M.db('WTD_r'), mi)).dropna()-Rtarget

Compute subsidy that results in this with partial equilibrium assumptions:

In [36]:
pzw = adj.rc_pd(M.db('pD'), M.get('n_ZW',m='W')).droplevel(('s','n'))
pmr = adj.rc_pd(M.db('p'), ni)
pwe = adj.rc_pd(M.db('pS'), M.db('n_wasteE')).droplevel(('s','n'))
Δγ  = adj.rc_pd(M.db('WTD_gr'), mi)-adj.rc_pd(M.db('WTD_ge'), mi)
ζtarget = adj.rc_pd(M.db('WTD_alphau') - (M.db('WTD_alphau')-M.db('WTD_alphal'))*(targetRate**(M.db('WTD_beta'))), mi).dropna()
τ = adjMultiIndex.bc(adj.rc_pd(1/(1+M.db('markup')), sIdx), mi)-(pzw*Δγ+adj.rc_pd(M.db('WTD_gwe'), mi) * adjMultiIndex.bc(pwe, mi))/(pmr*ζtarget)
τ = stdSort(τ.dropna()).droplevel('m').rename('tauS')

Add shock and solve:

In [37]:
di['shock'] = τ
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state = 'B')

Report solution:

In [38]:
di = Rep(di['db'], di, d0 = d0)

Revert model data:

In [39]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

## 5. Tax on waste generation

In [40]:
name = 'taxWasteGen' 
shocks[name] = {}
di = shocks[name] 

Create tax on inputs proportional to waste generation intensities:

In [41]:
uWS_adValorem = (M.db('uWS').groupby(['t','s','n']).sum() / M.db('p')).dropna() # unit tax per ton of waste across $m$ types

Tax per kton of waste:

In [42]:
di['shock'] = (uWS_adValorem * 0.1).rename('tauD')
M.db.aom(di['shock'], priority='second')
M.db.mergeInternal()
di['db'] = M.solve(state='B')

Report solution:

In [43]:
di = Rep(di['db'], di, d0 = d0)

Revert model data:

In [44]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

## 6. Subsidy to recycled goods 

In [45]:
name = 'subsidyRecycledInputs'
shocks[name] = {}
di = shocks[name]

Shock:

In [46]:
di['shock'] = adj.rc_pd(M.db('tauD'), Rep.rm)-0.25
M.db.aom(di['shock'], priority = 'second')
M.db.mergeInternal()
di['db'] = M.solve(state = 'B')

Report solution:

In [47]:
di = Rep(di['db'], di, d0 = d0)

Revert model data:

In [48]:
M.db.aom(adj.rc_pd(d0['db'](di['shock'].name), di['shock']))

## X. Export

### A. To excel

Collect in dataframes and export in sheets:

In [49]:
for name, di in shocks.items():
    domains = {k: tuple(v.index.names) for k,v in di.items() if isinstance(v, (pd.Series,pd.DataFrame))}
    uniqueCombinations = list(set(domains.values()))
    # Write to excel:
    with pd.ExcelWriter(os.path.join(d['curr'],'results',f'{name}.xlsx'), engine='openpyxl') as writer:
        dfs = {combi: pd.concat([di[k].rename(k) for k,v in domains.items() if v == combi], axis = 1) for combi in uniqueCombinations}
        for df_name, df in dfs.items():
            df.to_excel(writer, sheet_name=f"domains {''.join(list(df_name))}")

### B. To pickle

In [54]:
with open(os.path.join(d['curr'],'results', 'GRSIntRC_shocks'), "wb") as file:
    pickle.dump(shocks, file)